### **CI/CD DEMO - POLLING SCRIPT**
#### **TEAM 04 STUDENT 3 - TING ENG KIAT (S403)**
#### **Jupyter Notebook Version: 26 August 2026**

#### **STEP 01: SET-UP**

In [1]:
import boto3
import pandas as pd
import numpy as np
import time
from datetime import datetime

region = boto3.Session().region_name

BUCKET  = "nyp-26s1-iti113"
TEAM_ID = "team04"
PROJECT_NAME = "credit-card-fraud-detection"
PREFIX  = f"iti113/{TEAM_ID}/data/{PROJECT_NAME}"

PIPELINE_NAME = f"iti113-{TEAM_ID}-{PROJECT_NAME}"

BASELINE_KEY = f"{PREFIX}/raw/fraudTest.csv"        # original, never touched
ACTIVE_KEY   = f"{PREFIX}/raw/fraudTest_cicd.csv"   # pipeline's actual input
STAGING_KEY  = f"{PREFIX}/staging/candidate.csv"    # where new candidates land

s3 = boto3.client('s3', region_name=region)
sm = boto3.client('sagemaker', region_name=region)

print(f"Pipeline: {PIPELINE_NAME}")
print(f"Baseline: s3://{BUCKET}/{BASELINE_KEY}")
print(f"Active:   s3://{BUCKET}/{ACTIVE_KEY}")
print(f"Staging:  s3://{BUCKET}/{STAGING_KEY}")

Pipeline: iti113-team04-credit-card-fraud-detection
Baseline: s3://nyp-26s1-iti113/iti113/team04/data/credit-card-fraud-detection/raw/fraudTest.csv
Active:   s3://nyp-26s1-iti113/iti113/team04/data/credit-card-fraud-detection/raw/fraudTest_cicd.csv
Staging:  s3://nyp-26s1-iti113/iti113/team04/data/credit-card-fraud-detection/staging/candidate.csv


#### **STEP 02: PSI FUNCTION**

In [2]:
def psi(expected, actual, bins=10):
    breakpoints = np.percentile(expected, np.linspace(0, 100, bins + 1))
    breakpoints[0], breakpoints[-1] = -np.inf, np.inf
    e_pct = np.clip(np.histogram(expected, breakpoints)[0] / len(expected), 1e-4, None)
    a_pct = np.clip(np.histogram(actual, breakpoints)[0] / len(actual), 1e-4, None)
    return np.sum((a_pct - e_pct) * np.log(a_pct / e_pct))

PSI_THRESHOLD = 0.2
print("PSI function ready.")

PSI function ready.


#### **STEP 03: LOAD BASELINE DATA FOR PSI REFERENCE**

In [3]:
baseline_df = pd.read_csv(f"s3://{BUCKET}/{BASELINE_KEY}")
baseline_amt = baseline_df['amt']
print(f"Baseline rows: {len(baseline_df)}, mean amt: {baseline_amt.mean():.2f}")

Baseline rows: 555719, mean amt: 69.39


#### **STEP 04: FUNCTION TO CHECK STAGING FOR NEW UPLOADS**

In [4]:
def get_staging_last_modified():
    try:
        resp = s3.head_object(Bucket=BUCKET, Key=STAGING_KEY)
        return resp['LastModified']
    except s3.exceptions.ClientError:
        return None  # no file uploaded yet

last_seen_modified = get_staging_last_modified()
print(f"Current staging LastModified: {last_seen_modified}")

Current staging LastModified: 2026-08-26 08:09:14+00:00


#### **STEP 05: MAIN POLLING LOOP**

In [ ]:
poll_interval = 10  # seconds between checks
last_seen_modified = get_staging_last_modified()

print("Polling started. Waiting for new candidate uploads...")

while True:
    current_modified = get_staging_last_modified()

    if current_modified is not None and current_modified != last_seen_modified:
        print("=" * 50)
        print(f"[{datetime.now()}] New candidate detected in staging.")

        candidate_df = pd.read_csv(f"s3://{BUCKET}/{STAGING_KEY}")
        score = psi(baseline_amt, candidate_df['amt'])
        print(f"PSI score: {score:.3f}")

        if score > PSI_THRESHOLD:
            print("PSI exceeds threshold — promoting to active and triggering pipeline.")
            s3.copy_object(Bucket=BUCKET, CopySource=f"{BUCKET}/{STAGING_KEY}", Key=ACTIVE_KEY)
            sm.start_pipeline_execution(PipelineName=PIPELINE_NAME)
            print("Pipeline triggered.")
        else:
            print("PSI below threshold — no action taken.")

        last_seen_modified = current_modified

    time.sleep(poll_interval)

**RESULTS:**

**At the start:**

```
Polling started. Waiting for new candidate uploads...
```

**After uploading candidate_a.csv (PSI below threshold):**

```
Polling started. Waiting for new candidate uploads...
==================================================
[2026-08-25 01:23:11.026096] New candidate detected in staging.
PSI score: 0.000
PSI below threshold — no action taken.
```

**After uploading candidate_b.csv (PSI above threshold):**

```
Polling started. Waiting for new candidate uploads...
==================================================
[2026-08-25 01:23:11.026096] New candidate detected in staging.
PSI score: 0.000
PSI below threshold — no action taken.
==================================================
[2026-08-25 01:24:12.098471] New candidate detected in staging.
PSI score: 0.542
PSI exceeds threshold — promoting to active and triggering pipeline.
Pipeline triggered.
```